# 02 — Banking Fraud Detection: Data Preparation

Sigue directamente desde [`01_eda.ipynb`](./01_eda.ipynb). La idea de este notebook es traducir lo que salió del EDA en un dataset que ya se pueda modelar:

1. Deduplicar antes de dividir train/test — el EDA mostró que hacerlo después arriesgaba filtrar duplicados exactos entre ambos conjuntos.
2. Construir dos features simples (`hour_of_day`, `log_amount`) a partir de `Time` y `Amount`.
3. Hacer un split train/test cronológico (el dataset ya viene ordenado por `Time`), con el mismo criterio que en project 01.
4. Dejar `V1`–`V28` intactas, sin reglas de outliers — el EDA mostró que justo ahí vive la señal de fraude.
5. Dejar una demostración diagnóstica de las técnicas de desbalance (class weight, SMOTE) que se comparan formalmente en `03_modeling.ipynb`, sin persistir en disco un dataset sobremuestreado gigante.

In [1]:
import pandas as pd
import numpy as np
from imblearn.over_sampling import SMOTE

pd.set_option('display.max_columns', None)

RAW_PATH = '../data/creditcard.csv'
TRAIN_OUT = '../data/processed_train.csv'
TEST_OUT = '../data/processed_test.csv'

In [2]:
df = pd.read_csv(RAW_PATH)
print('Filas crudas:', df.shape)

Filas crudas: (284807, 31)


## 1. Deduplicar antes de dividir

Se elimina cualquier fila exactamente duplicada (todas las columnas iguales, incluida `Class`), y se conserva la primera aparición de cada una. Esto se hace antes del split, tal como quedó decidido en el EDA, para que ningún par duplicado termine repartido entre train y test.

In [3]:
n_before = len(df)
fraud_before = df['Class'].sum()

df = df.drop_duplicates(keep='first').reset_index(drop=True)

n_after = len(df)
fraud_after = df['Class'].sum()

print(f'Filas: {n_before} -> {n_after} (-{n_before - n_after})')
print(f'Fraudes: {fraud_before} -> {fraud_after} (-{fraud_before - fraud_after})')
print(f'Tasa de fraude: {fraud_before / n_before:.4%} -> {fraud_after / n_after:.4%}')

Filas: 284807 -> 283726 (-1081)
Fraudes: 492 -> 473 (-19)
Tasa de fraude: 0.1727% -> 0.1667%


## 2. Feature engineering

Dos features nada exóticas, pero ambas motivadas directamente por lo que salió del EDA.

`hour_of_day` es la hora del día (0–23) derivada de `Time`. El EDA mostró una tasa de fraude visiblemente más alta de madrugada, así que vale la pena dejársela al modelo de forma explícita.

`log_amount` es `log(1 + Amount)`. `Amount` viene muy sesgado a la derecha (ver EDA, sección 7), y el log ayuda al modelo lineal (el baseline) sin perjudicar a los modelos de árbol. Se conserva también `Amount` en su escala original, por si el modelo de árbol termina prefiriendo la escala cruda.

In [4]:
df['hour_of_day'] = (df['Time'] % 86400) // 3600
df['log_amount'] = np.log1p(df['Amount'])

df[['Time', 'hour_of_day', 'Amount', 'log_amount']].head()

,Time,hour_of_day,Amount,log_amount
0,0.0,0.0,149.62,5.014760
1,0.0,0.0,2.69,1.305626
2,1.0,0.0,378.66,5.939276
3,1.0,0.0,123.50,4.824306
4,2.0,0.0,69.99,4.262539


Sobre `V1`–`V28`: no se aplica ninguna transformación ni recorte de outliers, quedan tal cual las entrega el dataset — consistente con la decisión tomada en el EDA (sección 8).

Sobre el escalado: `Time` y `Amount`/`log_amount` no están en la misma escala que `V1`–`V28` (que ya vienen centradas por el PCA del proveedor). Ese escalado se deja para `03_modeling.ipynb`, dentro de un `Pipeline` de scikit-learn ajustado únicamente sobre train — el mismo criterio de project 01, para no filtrar estadísticas del test set.

## 3. Split train/test cronológico

El dataset ya está ordenado por `Time`, esto se verificó en el EDA. Se separa el 80% más antiguo para train y el 20% más reciente para test, mismo criterio que en project 01: en producción, un modelo de fraude siempre predice sobre transacciones que todavía no ocurrieron, nunca sobre el pasado.

In [5]:
cutoff_time = df['Time'].quantile(0.8)

train_df = df[df['Time'] < cutoff_time].copy()
test_df = df[df['Time'] >= cutoff_time].copy()

print(f'Corte en Time = {cutoff_time:.0f}s (~{cutoff_time / 3600:.1f}h)')
print(f'Train: {len(train_df)} filas | fraude: {train_df["Class"].sum()} ({train_df["Class"].mean():.4%})')
print(f'Test:  {len(test_df)} filas | fraude: {test_df["Class"].sum()} ({test_df["Class"].mean():.4%})')

Corte en Time = 145234s (~40.3h)


Train: 226980 filas | fraude: 399 (0.1758%)
Test:  56746 filas | fraude: 74 (0.1304%)


Ambos conjuntos terminan con una tasa de fraude parecida (~0.13–0.18%) y un volumen de casos positivos razonable: 399 en train, 74 en test. Alcanza para entrenar y evaluar, aunque en términos absolutos siguen siendo pocos casos — vale tenerlo presente al interpretar cualquier métrica más adelante.

## 4. Diagnóstico de técnicas de desbalance (sin persistir nada todavía)

El EDA había marcado tres técnicas para evaluar en el modelado: class weights, SMOTE e Isolation Forest (esta última no supervisada). Acá solo dejo un diagnóstico rápido de las dos primeras — la comparación formal de desempeño ocurre recién en `03_modeling.ipynb`. Y a propósito no se guarda ningún CSV sobremuestreado: SMOTE se recalcula al vuelo en el notebook de modelado, ajustado únicamente sobre el split de train de cada corrida, para no perder trazabilidad de qué filas terminan siendo sintéticas.

In [6]:
X_train_preview = train_df.drop(columns=['Class'])
y_train_preview = train_df['Class']

neg, pos = (y_train_preview == 0).sum(), (y_train_preview == 1).sum()
scale_pos_weight = neg / pos
print(f'class weight (negativas/positivas) sugerido para XGBoost: scale_pos_weight = {scale_pos_weight:.1f}')

X_res, y_res = SMOTE(random_state=42).fit_resample(X_train_preview, y_train_preview)
print(f'SMOTE — antes: {y_train_preview.value_counts().to_dict()} | después: {y_res.value_counts().to_dict()}')

class weight (negativas/positivas) sugerido para XGBoost: scale_pos_weight = 567.9


SMOTE — antes: {0: 226581, 1: 399} | después: {0: 226581, 1: 226581}


Un desbalance de ~568:1 es bastante más severo de lo que uno se acostumbra a ver, y `scale_pos_weight` termina siendo un ajuste enorme. En `03_modeling.ipynb` esto se compara contra SMOTE (que genera ejemplos sintéticos de fraude por interpolación) y contra Isolation Forest (que no necesita ninguna de las dos técnicas, al ser no supervisado). Ninguna se descarta de antemano acá — la comparación real, con PR-AUC como criterio, queda para el notebook de modelado.

## 5. Guardado de datasets procesados

Los datasets se guardan deduplicados, con las dos features nuevas, pero sin escalar y sin resamplear. El resampleo y el escalado quedan dentro del pipeline de modelado, no horneados de antemano en el CSV. No se versionan en git — están excluidos por `.gitignore`, igual que el CSV crudo.

In [7]:
train_df.to_csv(TRAIN_OUT, index=False)
test_df.to_csv(TEST_OUT, index=False)

print('Guardado:', TRAIN_OUT, train_df.shape)
print('Guardado:', TEST_OUT, test_df.shape)

Guardado: ../data/processed_train.csv (226980, 33)
Guardado: ../data/processed_test.csv (56746, 33)


## 6. Cómo queda el dataset después de esta fase

Se deduplicó antes del split: 1.081 filas exactas eliminadas (19 de fraude), evitando fuga de información entre train y test por duplicación.

Se agregaron dos features nuevas: `hour_of_day` (por el patrón horario que apareció en el EDA) y `log_amount` (para corregir el sesgo de `Amount`).

`V1`–`V28` quedaron intactas, sin ningún recorte de outliers, por la decisión explícita que se tomó en el EDA.

El split es cronológico, 80/20 por `Time`: train queda con 399 fraudes, test con 74. Así se evalúa predicción hacia adelante, no interpolación sobre datos ya vistos.

El desbalance queda diagnosticado pero no resuelto acá (`scale_pos_weight` ≈ 568, SMOTE balancea a 226.581/226.581 en cada clase) — comparar class weights, SMOTE e Isolation Forest es tarea de `03_modeling.ipynb`, con PR-AUC como criterio, no accuracy.

Y el escalado queda diferido: `Time`/`Amount` se escalan recién dentro del pipeline de modelado, ajustado solo sobre train.

Siguiente paso: `03_modeling.ipynb`, donde se comparan Logistic Regression, XGBoost e Isolation Forest bajo las distintas estrategias de desbalance, usando PR-AUC y la curva precision-recall como criterio principal.